In [1]:
%load_ext autoreload
%autoreload 2

# Walkthrough Example

This document provides a walkthrough of how to use the `oceanicospy` package to handle observations from various sensors.

# Importing libraries

All the sensors are represented as objects in the module `oceanicospy.observations`.

In [2]:
from oceanicospy.observations import pressure_sensors,weather_stations,AWAC,Buoy,ctd
from datetime import datetime, timedelta
import glob

# Pressure sensors: AQUALogger, RBR, BlueLogger

## Data directories and sampling configuration

For every sensor, we specify the directory where the data files are located and a sampling configuration dictionary that contains parameters such as anchoring depth, sensor height, sampling frequency, burst length, and optional start and end times for filtering the data.

In [3]:
measurement_pressure_sensors_paths = ['../data/observations/AQUAlogger/','../data/observations/RBR/','../data/observations/Bluelog/']

sampling_AQ = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=1,burst_length_s=2048,
                    start_time=datetime(2025,5,9,10,0,0),end_time=datetime(2025,5,19,18,0,0)-timedelta(seconds=1))
sampling_RBR = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=2, burst_length_s=7200,
                    start_time=datetime(2025,5,9,10,0,0),end_time=datetime(2025,5,19,18,0,0)-timedelta(seconds=0.5))
sampling_Bluelog = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=2, burst_length_s=4096, temperature=False,
                        start_time=datetime(2025,7,4,14,0,0), end_time=datetime(2025,7,10,12,0,0)-timedelta(seconds=1))

## Reading and storing data

In [4]:

sampling_data = [sampling_AQ,sampling_RBR,sampling_Bluelog]
metadata_list=['AQ','RBR','Bluelog']
dict_raw_measurements = dict()
dict_clean_measurements = dict() 

for idx,measurement_path in enumerate(measurement_pressure_sensors_paths):
    print(measurement_path)
    if 'AQ' in measurement_path:
        object_device = pressure_sensors.AQUAlogger(measurement_path, sampling_data[idx],filename='AQUAlogger_520PT5.csv')
    elif 'Bluelog' in measurement_path:
        object_device = pressure_sensors.Bluelog(measurement_path, sampling_data[idx])
    else:
        object_device = pressure_sensors.RBR(measurement_path,sampling_data[idx])
    raw_data = object_device.get_raw_records()
    clean_data = object_device.get_clean_records()
    dict_raw_measurements[metadata_list[idx]] = raw_data
    dict_clean_measurements[metadata_list[idx]] = clean_data

../data/observations/AQUAlogger/
../data/observations/RBR/
../data/observations/Bluelog/


A quick look at the data dictionary can show the available data for each sensor. Each sensor's data is retrieved as a pandas DataFrame, which allows for easy manipulation and analysis.

In [5]:
dict_clean_measurements['AQ']

,pressure[bar],depth[m],depth_aux[m],burstId,eta[m]
date,,,,,
2025-05-09 10:00:00,1.245722,3.136509,2.307327,1,0.052596
2025-05-09 10:00:01,1.244709,3.124966,2.297273,1,0.041052
2025-05-09 10:00:02,1.237181,3.039040,2.222556,1,-0.044874
2025-05-09 10:00:03,1.243695,3.113418,2.287209,1,0.029503
2025-05-09 10:00:04,1.253249,3.222086,2.382034,1,0.138170
...,...,...,...,...,...
2025-05-19 16:34:03,1.239498,3.065513,2.245553,124,-0.003209
2025-05-19 16:34:04,1.242248,3.096910,2.272847,124,0.028177
2025-05-19 16:34:05,1.237905,3.047316,2.229742,124,-0.021427


In [6]:
dict_clean_measurements['RBR']

,pressure[bar],depth[m],burstId,eta[m]
date,,,,
2025-05-09 10:00:00.000,1.241829,2.267149,1,0.117985
2025-05-09 10:00:00.500,1.239702,2.246051,1,0.096886
2025-05-09 10:00:01.000,1.235314,2.202530,1,0.053365
2025-05-09 10:00:01.500,1.233978,2.189276,1,0.040110
2025-05-09 10:00:02.000,1.234466,2.194119,1,0.044954
...,...,...,...,...
2025-05-19 17:59:57.500,1.228489,2.134836,248,-0.044336
2025-05-19 17:59:58.000,1.228343,2.133389,248,-0.045786
2025-05-19 17:59:58.500,1.226372,2.113838,248,-0.065340


In [7]:
dict_clean_measurements['Bluelog']

,pressure[bar],temperature[c],depth[m],burstId,eta[m]
date,,,,,
2025-07-04 14:00:00.000,1.0615,27.80,0.478890,1,-0.000014
2025-07-04 14:00:00.500,1.0616,27.80,0.479883,1,0.000980
2025-07-04 14:00:01.000,1.0617,27.80,0.480875,1,0.001974
2025-07-04 14:00:01.500,1.0617,27.80,0.480875,1,0.001976
2025-07-04 14:00:02.000,1.0615,27.80,0.478890,1,-0.000008
...,...,...,...,...,...
2025-07-10 11:34:05.500,2.1055,27.65,10.840782,142,0.014486
2025-07-10 11:34:06.000,2.1045,27.65,10.830857,142,0.004558
2025-07-10 11:34:06.500,2.1037,27.65,10.822917,142,-0.003385


# ADCP sensors: AWAC

Support for ADCP sensors, such as the AWAC, is also included in the module. The `AWAC` class is designed to handle the specific data formats and requirements of ADCP measurements, allowing users to easily load and process wave records from AWAC deployments.

Similar to the pressure sensors, the `AWAC` class also requires a directory path and sampling configuration for initialization. 

In [3]:
measurement_AWAC_paths = ['../data/observations/AWAC/']

sampling_AWAC=dict(anchoring_depth=1,sensor_height=0.2,sampling_freq=1,burst_length_s=2048,temperature=False,
                            start_time=datetime(2023,8,18,19,0,0),end_time=datetime(2023,9,1,15,0,0)-timedelta(seconds=1))

In [4]:
help(AWAC)

Help on class AWAC in module oceanicospy.observations.awac:

class AWAC(builtins.object)
 |  AWAC(directory_path, sampling_data)
 |  
 |  A class to handle reading and processing the data files recorded by an ADCP AWAC (Nortek S.A). 
 |  
 |  Parameters
 |  ----------
 |  directory_path : str
 |      Path to the directory containing the .hdr and .wad files.
 |  sampling_data : dict
 |      Dictionary containing the information about the device installation
 |  
 |  Notes
 |  -----
 |  
 |  - 04-Jan-2018 : Origination - Daniel Peláez
 |  - 01-Sep-2023 : Migration to Python - Alejandro Henao
 |  - 10-Dec-2024 : Class implementation - Franklin Ayala
 |  
 |  Methods defined here:
 |  
 |  __init__(self, directory_path, sampling_data)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  get_clean_currents_records(self, compute_speed_dir=True)
 |      Processes the raw current data by reading the x and y components, setting the index to a date range, and optionall

In [5]:
AWAC_obj = AWAC(measurement_AWAC_paths[0],sampling_AWAC)

## Reading wave records

The method `get_raw_wave_records` can be used to retrieve the raw wave records, with an option to specify whether to read from a single WAD file or multiple files in the directory.

### Using the.wad file with all the records

In [33]:
df_raw_wave_single_wad = AWAC_obj.get_raw_wave_records(from_single_wad=True)
df_clean_wave_single_wad = AWAC_obj.get_clean_wave_records(from_single_wad=True)

In [34]:
df_raw_wave_single_wad

,month,day,year,hour,minute,second,3 Pressure (dbar),4 Spare,5 Analog input,6 Velocity (Beam1) (m/s),7 Velocity (Beam2) (m/s),8 Velocity (Beam3) (m/s),9 Velocity (Beam4) (m/s),10 Amplitude (Beam1) (counts),11 Amplitude (Beam2) (counts),12 Amplitude (Beam3) (counts),13 Amplitude (Beam4) (counts)
0,8,18,2023,19,0,1.0,0.000,13.286,14.561,13,0,-1.780,-0.601,-3.245,24,24,20
1,8,18,2023,19,0,1.5,0.000,14.707,12.990,14,0,-0.173,1.524,-4.112,22,23,18
2,8,18,2023,19,0,2.0,0.000,13.975,15.950,12,0,2.813,2.377,-3.706,24,24,19
3,8,18,2023,19,0,2.5,0.000,14.523,13.978,14,0,-2.034,-2.001,2.567,23,24,19
4,8,18,2023,19,0,3.0,0.000,10.388,16.318,14,0,-1.913,0.129,2.931,23,23,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
679931,9,1,2023,14,17,2.5,0.000,15.761,11.693,19,0,-0.791,-1.898,1.816,21,22,18
679932,9,1,2023,14,17,3.0,0.001,15.654,12.824,13,0,-3.340,0.069,4.258,22,22,18
679933,9,1,2023,14,17,3.5,0.000,11.113,15.252,10,0,-0.742,-0.419,1.156,21,21,18
679934,9,1,2023,14,17,4.0,0.002,13.611,12.247,14,0,-3.187,-2.641,2.729,22,22,18


In [35]:
df_clean_wave_single_wad

,pressure[bar],velocitybeam1[m/s],velocitybeam2[m/s],velocitybeam3[m/s],velocitybeam4[m/s],burstId
date,,,,,,
2023-08-18 19:00:01.000,0.0000,13,0,-1.780,-0.601,1
2023-08-18 19:00:01.500,0.0000,14,0,-0.173,1.524,1
2023-08-18 19:00:02.000,0.0000,12,0,2.813,2.377,1
2023-08-18 19:00:02.500,0.0000,14,0,-2.034,-2.001,1
2023-08-18 19:00:03.000,0.0000,14,0,-1.913,0.129,1
...,...,...,...,...,...,...
2023-09-01 14:17:02.500,0.0000,19,0,-0.791,-1.898,332
2023-09-01 14:17:03.000,0.0001,13,0,-3.340,0.069,332
2023-09-01 14:17:03.500,0.0000,10,0,-0.742,-0.419,332


### Using .wad files recorded per burst

In [30]:
df_raw_wave_combined_wad = AWAC_obj.get_raw_wave_records(from_single_wad=False)
df_clean_wave_combined_wad = AWAC_obj.get_clean_wave_records(from_single_wad=False)

In [31]:
df_raw_wave_combined_wad

,burstId,2 Ensemble counter,3 Pressure (dbar),4 Spare,5 Analog input,6 Velocity (Beam1) (m/s),7 Velocity (Beam2) (m/s),8 Velocity (Beam3) (m/s),9 Velocity (Beam4) (m/s),10 Amplitude (Beam1) (counts),11 Amplitude (Beam2) (counts),12 Amplitude (Beam3) (counts),13 Amplitude (Beam4) (counts)
0,1,1,0.000,13.286,14.561,13,0,-1.780,-0.601,-3.245,24,24,20
1,1,2,0.000,14.707,12.990,14,0,-0.173,1.524,-4.112,22,23,18
2,1,3,0.000,13.975,15.950,12,0,2.813,2.377,-3.706,24,24,19
3,1,4,0.000,14.523,13.978,14,0,-2.034,-2.001,2.567,23,24,19
4,1,5,0.000,10.388,16.318,14,0,-1.913,0.129,2.931,23,23,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...
679931,332,2044,0.000,15.761,11.693,19,0,-0.791,-1.898,1.816,21,22,18
679932,332,2045,0.001,15.654,12.824,13,0,-3.340,0.069,4.258,22,22,18
679933,332,2046,0.000,11.113,15.252,10,0,-0.742,-0.419,1.156,21,21,18
679934,332,2047,0.002,13.611,12.247,14,0,-3.187,-2.641,2.729,22,22,18


In [32]:
df_clean_wave_combined_wad

,pressure[bar],burstId,velocitybeam1[m/s],velocitybeam2[m/s],velocitybeam3[m/s],velocitybeam4[m/s]
2023-08-18 19:00:00.000,0.0000,1,13,0,-1.780,-0.601
2023-08-18 19:00:00.500,0.0000,1,14,0,-0.173,1.524
2023-08-18 19:00:01.000,0.0000,1,12,0,2.813,2.377
2023-08-18 19:00:01.500,0.0000,1,14,0,-2.034,-2.001
2023-08-18 19:00:02.000,0.0000,1,14,0,-1.913,0.129
...,...,...,...,...,...,...
2023-09-01 14:17:01.500,0.0000,332,19,0,-0.791,-1.898
2023-09-01 14:17:02.000,0.0001,332,13,0,-3.340,0.069
2023-09-01 14:17:02.500,0.0000,332,10,0,-0.742,-0.419
2023-09-01 14:17:03.000,0.0002,332,14,0,-3.187,-2.641


## Reading current records

In [65]:
x_currents_raw,y_currents_raw = AWAC_obj.get_raw_currents_records()

In [66]:
x_currents_raw.head()

,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,...,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,cell_23,cell_24,cell_25,cell_26
0,5.858,2.155,1.184,-0.752,-0.320,-0.186,-0.531,-0.454,-0.264,-0.505,...,-0.765,-0.718,-0.466,-0.651,-0.516,-0.760,-0.500,-0.583,-0.531,-0.792
1,5.066,1.388,1.375,-0.306,-0.914,-1.109,-0.326,-0.781,-0.969,-0.845,...,-0.787,-0.957,-0.992,-0.718,-0.924,-0.924,-1.162,-1.027,-0.815,-0.901
2,5.291,1.272,1.047,-0.908,-1.080,-0.526,-0.659,-0.788,-0.857,-0.782,...,-0.633,-0.848,-0.946,-1.212,-0.807,-1.084,-1.228,-0.706,-0.674,-0.703
3,5.837,1.354,0.851,-1.224,-0.305,-0.741,-0.887,-0.402,-1.112,-0.538,...,-0.980,-0.733,-0.872,-0.713,-0.859,-0.808,-1.001,-0.843,-0.967,-0.490
4,5.455,1.276,0.416,-0.563,-0.553,-0.836,-0.744,-1.075,-0.388,-1.072,...,-0.552,-1.110,-0.557,-1.027,-0.638,-0.314,-1.031,-0.830,-0.714,-1.002


In [67]:
y_currents_raw.head()

,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,...,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,cell_23,cell_24,cell_25,cell_26
0,0.581,0.683,0.058,0.720,0.583,0.744,0.745,0.482,1.127,0.846,...,0.436,0.984,0.659,0.216,0.487,0.727,0.896,0.307,0.940,0.801
1,0.131,0.243,0.087,0.867,0.647,0.483,0.850,0.916,0.368,0.921,...,0.300,0.861,0.946,0.366,1.261,1.377,0.966,0.890,0.688,1.216
2,0.219,0.181,-0.189,0.769,1.040,0.835,0.922,0.643,0.476,0.943,...,0.772,0.912,1.036,0.672,0.990,0.849,0.314,1.008,1.130,0.921
3,0.423,0.087,0.289,0.432,0.807,0.773,0.303,0.590,0.488,0.548,...,0.870,0.685,1.256,0.623,0.547,0.933,0.774,0.847,0.526,0.689
4,0.278,-0.048,-0.105,0.834,0.659,0.616,0.965,0.971,1.076,0.901,...,0.915,0.953,0.979,0.990,0.985,0.521,1.035,0.979,0.487,0.837


In [68]:
help(AWAC.get_clean_currents_records)

Help on function get_clean_currents_records in module oceanicospy.observations.awac:

get_clean_currents_records(self, compute_speed_dir=True)
    Processes the raw current data by reading the x and y components, setting the index to a date range, and optionally computing speed and direction.
    
    Parameters
    ----------
    compute_speed_dir : bool, optional
        If True, computes the current speed and direction from the x and y components. Default is True.
    
    Returns
    -------
    x_component_clean : pandas.DataFrame
        Cleaned DataFrame of the x component of the current.
    y_component_clean : pandas.DataFrame
        Cleaned DataFrame of the y component of the current.
    current_speed : pandas.DataFrame, optional
        DataFrame of the current speed, only returned if compute_speed_dir is True.
    current_dir : pandas.DataFrame, optional
        DataFrame of the current direction, only returned if compute_speed_dir is True.



The method `get_clean_currents_records` allows users to retrieve cleaned current records, which can be useful for further analysis and interpretation of the data. If the `compute_speed_dir` parameter is set to `True`, the method will also compute the current speed and direction based on the u and v components of the currents.

In [69]:
x_currents_clean,y_currents_clean,current_speed,current_dir = AWAC_obj.get_clean_currents_records()

/Users/franklinayala/Documents/projects/oceanicospy/oceanicospy/utils/wave_props.py:84: RuntimeWarning: invalid value encountered in scalar divide
  theta = 90 + (np.arctan(abs(y/x))*(180/np.pi))


In [70]:
x_currents_clean

,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,...,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,cell_23,cell_24,cell_25,cell_26
2023-08-18 19:25:00,5.066,1.388,1.375,-0.306,-0.914,-1.109,-0.326,-0.781,-0.969,-0.845,...,-0.787,-0.957,-0.992,-0.718,-0.924,-0.924,-1.162,-1.027,-0.815,-0.901
2023-08-18 19:55:00,5.291,1.272,1.047,-0.908,-1.080,-0.526,-0.659,-0.788,-0.857,-0.782,...,-0.633,-0.848,-0.946,-1.212,-0.807,-1.084,-1.228,-0.706,-0.674,-0.703
2023-08-18 20:25:00,5.837,1.354,0.851,-1.224,-0.305,-0.741,-0.887,-0.402,-1.112,-0.538,...,-0.980,-0.733,-0.872,-0.713,-0.859,-0.808,-1.001,-0.843,-0.967,-0.490
2023-08-18 20:55:00,5.455,1.276,0.416,-0.563,-0.553,-0.836,-0.744,-1.075,-0.388,-1.072,...,-0.552,-1.110,-0.557,-1.027,-0.638,-0.314,-1.031,-0.830,-0.714,-1.002
2023-08-18 21:25:00,5.852,1.275,0.659,-1.052,-0.946,-0.900,-0.785,-0.626,-0.889,-0.693,...,-0.942,-1.013,-0.991,-0.724,-0.940,-0.756,-0.713,-0.700,-0.407,-0.323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-09-01 11:55:00,-2.968,1.617,1.326,1.148,0.776,0.758,1.178,0.826,1.249,1.242,...,0.954,1.003,0.662,0.631,1.246,1.237,1.251,1.238,0.738,1.336
2023-09-01 12:25:00,-3.214,0.663,0.887,0.512,0.630,0.491,0.647,0.660,0.552,0.860,...,1.028,0.961,0.339,0.337,0.521,0.782,0.552,0.683,1.069,0.610
2023-09-01 12:55:00,2.366,-0.173,0.948,0.771,0.332,0.679,0.327,0.609,0.775,0.326,...,0.720,0.994,1.064,0.460,0.193,0.300,0.414,0.290,0.260,0.621
2023-09-01 13:25:00,2.539,0.120,1.091,0.947,1.514,1.295,1.538,1.416,1.248,1.550,...,1.061,1.288,1.254,1.167,1.223,1.196,1.205,1.139,1.663,1.088


In [71]:
current_dir

,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,...,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,cell_23,cell_24,cell_25,cell_26
2023-08-18 18:55:00,84.335894,72.414647,87.195523,313.754636,331.238274,345.963757,324.520564,316.713465,346.816166,329.165894,...,299.680314,323.882713,324.734646,288.355702,313.343854,313.728682,330.836922,297.770691,330.538172,315.323702
2023-08-18 19:25:00,88.518738,80.069749,86.379566,340.559965,305.293817,293.534450,339.016712,319.548424,290.795423,317.464210,...,290.866567,311.977288,313.640294,297.010181,323.767836,326.137442,309.737605,310.912259,310.170113,323.463258
2023-08-18 19:55:00,87.629821,81.901431,100.232595,310.261807,313.919076,327.791528,324.444691,309.214096,299.048927,320.332159,...,320.650023,317.082565,317.599940,299.006416,320.814750,308.068414,284.343221,324.992723,329.185606,322.645478
2023-08-18 20:25:00,85.855093,86.323567,71.242489,289.440035,339.296266,316.210827,288.860213,325.731207,293.694216,315.527571,...,311.597230,313.061250,325.229008,311.146077,302.488453,319.106665,307.712156,315.135611,298.543996,324.580490
2023-08-18 20:55:00,87.082592,92.154311,104.165798,325.978290,319.998299,306.384352,322.368335,312.090109,340.170960,310.046550,...,328.898315,310.648022,330.362379,313.949080,327.068235,328.923170,315.110931,319.708592,304.296836,309.872998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-09-01 11:55:00,293.572642,31.662334,66.738860,68.942425,79.916221,82.113438,79.327494,64.668735,108.090213,77.339377,...,78.678518,81.215208,83.109452,87.006276,64.436781,79.109319,78.171369,83.776666,60.296940,83.551570
2023-09-01 12:25:00,227.753830,53.813657,54.615372,42.950743,52.011858,48.200303,59.639059,59.549633,53.597949,71.007252,...,70.650392,66.844597,40.708903,28.143246,48.434792,48.990913,59.357868,38.423689,74.026292,67.714412
2023-09-01 12:55:00,84.303793,183.882696,149.290325,145.835372,166.575951,135.335550,166.631899,154.847827,148.667024,163.232766,...,154.798876,147.809078,139.276767,160.204299,172.479813,163.731235,162.011081,162.575388,168.544146,148.590948
2023-09-01 13:25:00,153.189143,177.983376,151.662101,130.750437,117.525645,128.551734,132.755773,125.766676,130.129171,118.597509,...,137.055622,110.828556,126.972107,106.197970,132.227018,117.439728,126.968670,128.592264,124.827517,131.394043


# Weather stations: Davis Vantage Pro, WeatherSens and Rainwise

Currently, there is support for three different weather station models: Davis Vantage Pro, WeatherSens and Rainwise

In [43]:
Davis_weather_station = weather_stations.DavisVantagePro('../data/observations/WeatherStations/DavisVantagePro.txt')
WeatherSens_weather_station = weather_stations.WeatherSens('../data/observations/WeatherStations/WeatherSens.xlsx')
Rainwise_weather_station = weather_stations.Rainwise('../data/observations/WeatherStations/Rainwise.csv')

object_list = [Davis_weather_station,WeatherSens_weather_station,Rainwise_weather_station]

In [44]:
dict_raw_measurements = dict()
dict_clean_measurements = dict()

for ws in object_list:
    raw_data = ws.get_raw_records()
    clean_data = ws.get_clean_records()
    dict_raw_measurements[ws.__class__.__name__] = raw_data
    dict_clean_measurements[ws.__class__.__name__] = clean_data

/Users/franklinayala/Documents/projects/oceanicospy/oceanicospy/observations/weather_stations/davis.py:78: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('---', np.nan, inplace=True)
/Users/franklinayala/venvs/oceanicospy-dev-ffayalac/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/franklinayala/venvs/oceanicospy-dev-ffayalac/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [46]:
dict_clean_measurements['DavisVantagePro']

,Temp1,Temp2,Speed,Dir1,Run,Speed2,Dir2,Bar,Rain,Rate,Temp4,Hum2,Dew,Heat,EMC,Density,Samp,Tx,Recept,Int.
date,,,,,,,,,,,,,,,,,,,,
2023-08-19 00:15:00,NaN,NaN,0.0,NaN,0.00,0.0,NaN,1011.7,0.00,0.0,28.8,79,24.8,34.2,15.47,1.1319,0,1,0.0,15
2023-08-19 00:30:00,NaN,NaN,0.0,NaN,0.00,0.0,NaN,1011.6,0.00,0.0,28.7,78,24.4,33.7,15.18,1.1329,0,1,0.0,15
2023-08-19 00:45:00,NaN,NaN,0.0,NaN,0.00,0.0,NaN,1011.3,0.00,0.0,28.4,78,24.2,33.1,15.19,1.1339,0,1,0.0,15
2023-08-19 01:00:00,NaN,NaN,0.0,NaN,0.00,0.0,NaN,1011.1,0.00,0.0,28.6,80,24.8,33.8,15.81,1.1321,0,1,0.0,15
2023-08-19 01:15:00,NaN,NaN,0.0,NaN,0.00,0.0,NaN,1010.8,0.00,0.0,28.7,81,25.1,34.3,16.21,1.1307,0,1,0.0,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-31 15:30:00,NaN,NaN,0.4,NNE,0.40,4.0,NNE,1008.9,0.00,0.0,32.2,77,27.7,43.9,14.55,1.1094,292,1,85.4,15
2023-08-31 15:45:00,NaN,NaN,1.8,NNE,1.61,4.5,ENE,1008.8,0.00,0.0,32.2,76,27.5,43.4,14.25,1.1098,295,1,86.3,15
2023-08-31 16:00:00,NaN,NaN,4.5,ENE,4.02,7.6,ENE,1008.8,0.00,0.0,32.2,76,27.5,43.4,14.25,1.1097,282,1,82.5,15


In [47]:
dict_clean_measurements['WeatherSens']

,Device Restarts,Rain,Speed,Direction,Temp,Hum,Bar,Radiacion Solar (W/m2),Bateria (V),Panel Supply (V),DBA (dBm)
date,,,,,,,,,,,
1999-12-31 19:15:00,NaN,0.0,3.2,80.400002,30.500000,82.199997,1010.099976,762.700012,11.53,0.070000,-51.0
1999-12-31 19:30:00,NaN,0.0,2.6,87.500000,30.600000,82.199997,1010.200012,776.000000,11.52,0.050000,-51.0
1999-12-31 19:45:00,NaN,0.0,2.0,75.400002,30.799999,81.800003,1010.299988,812.099976,11.51,0.060000,-51.0
1999-12-31 20:00:00,NaN,0.0,2.4,73.900002,31.200001,80.599998,1010.200012,474.600006,11.51,0.070000,-51.0
1999-12-31 20:15:00,NaN,0.0,2.1,66.300003,31.200001,80.699997,1010.200012,742.599976,11.49,0.050000,-51.0
...,...,...,...,...,...,...,...,...,...,...,...
2025-07-10 15:45:00,NaN,0.0,4.0,49.099998,29.299999,82.500000,1009.000000,0.000000,13.73,19.990000,-51.0
2025-07-10 16:00:00,NaN,0.0,3.8,51.400002,29.299999,83.099998,1009.000000,0.000000,13.73,20.020000,-51.0
2025-07-10 16:15:00,NaN,0.0,5.5,45.200001,29.299999,83.300003,1008.799988,0.000000,13.74,19.020000,-51.0


In [48]:
dict_clean_measurements['Rainwise']

,Time,Temp Avg,Temp Low,Temp High,Heat Index,Wind Chill,Temp (Day) Low,Temp (Day) High,Hum Avg,Hum Low,...,Inside Temp (Day) High,Station Voltage,UV Index,Temp 1 Avg,Temp 1 Low,Temp 1 High,Solar Radiation 2 Avg,Solar Radiation 2 Sum (Interval),Solar Radiation 2 Sum (Day),Battery Voltage 2
0,2023-11-09 11:59:14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-11-09 12:00:00,31.4,NaN,NaN,45.7,31.4,NaN,NaN,88.0,NaN,...,NaN,6.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.9
2,2023-11-09 12:15:00,33.9,NaN,NaN,52.6,33.9,NaN,NaN,81.0,NaN,...,NaN,6.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.9
3,2023-11-09 12:31:07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-11-09 12:45:00,34.8,NaN,NaN,39.2,34.8,NaN,NaN,47.0,NaN,...,NaN,6.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
904,2023-11-18 22:00:00,29.1,NaN,NaN,33.4,29.1,NaN,NaN,72.0,NaN,...,NaN,5.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.9
905,2023-11-18 22:15:00,29.3,NaN,NaN,33.3,29.3,NaN,NaN,69.0,NaN,...,NaN,5.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.9
906,2023-11-18 22:30:00,29.3,NaN,NaN,33.2,29.3,NaN,NaN,69.0,NaN,...,NaN,5.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.9
907,2023-11-18 22:45:00,29.0,NaN,NaN,32.7,29.0,NaN,NaN,70.0,NaN,...,NaN,5.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.9


# Spotter buoy

In [39]:
csv_files_spotter = glob.glob('../data/observations/Buoy/*.csv')
dict_spotter_sources = dict()
for csv_file in csv_files_spotter:
    spotter_object = Buoy(csv_file)
    dict_spotter_sources[csv_file.split('/')[-1].split('.')[0]] = spotter_object.get_raw_records()

In [41]:
dict_spotter_sources['spotter_data_aqualink']

,timestamp,wind_speed_gfs,wind_direction_gfs,temp_alert_noaa,temp_weekly_alert_noaa,dhw_noaa,satellite_temperature_noaa,sst_anomaly_noaa,top_temperature_spotter,bottom_temperature_spotter,significant_wave_height_spotter,wave_mean_period_spotter,wave_mean_direction_spotter,wind_speed_spotter,wind_direction_spotter,significant_wave_height_sofar_model,wave_mean_period_sofar_model,wave_mean_direction_sofar_model
date,,,,,,,,,,,,,,,,,,
2022-05-01 01:00:00,2022-05-01T06:00:00.000Z,5.116343,24.501352,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.484210,5.385854,70.847834
2022-05-01 07:00:00,2022-05-01T12:00:00.000Z,6.465680,27.656962,0.0,0.0,0.000000,27.570000,0.060666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.387397,5.392312,69.502012
2022-05-01 13:00:00,2022-05-01T18:00:00.000Z,5.919429,40.296021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.339071,5.306848,68.041002
2022-05-01 19:00:00,2022-05-02T00:00:00.000Z,7.152353,29.802939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.361218,5.088913,62.355534
2022-05-02 01:00:00,2022-05-02T06:00:00.000Z,6.601567,40.623034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.426305,5.052871,59.526641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-27 07:00:00,2023-08-27T12:00:00.000Z,NaN,NaN,2.0,2.0,3.359647,29.899956,1.791569,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-28 07:00:00,2023-08-28T12:00:00.000Z,NaN,NaN,2.0,2.0,3.399530,30.049994,1.930962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-29 07:00:00,2023-08-29T12:00:00.000Z,NaN,NaN,2.0,2.0,3.439877,29.979915,1.850237,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
dict_spotter_sources['spotter_data_sofar']

,Battery Voltage (V),Power (W),Humidity (%rel),Epoch Time,Significant Wave Height (m),Peak Period (s),Mean Period (s),Peak Direction (deg),Peak Directional Spread (deg),Mean Direction (deg),...,Partition0 Mean Direction (deg),Partition0 Mean Directional Spread (deg),Partition1 Start Frequency (hz),Partition1 End Frequency (hz),Partition1 Significant Wave Height (m),Partition1 Mean Period (s),Partition1 Mean Direction (deg),Partition1 Mean Directional Spread (deg),Mean Barometric Pressure (hPa),Processing Source
date,,,,,,,,,,,,,,,,,,,,,
2025-05-01 00:40:00,4.01,-0.14,81.6,1746078000,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,embedded
2025-05-01 01:10:00,4.01,-0.14,81.6,1746079800,0.634,6.827,3.221,81.378,69.047,58.304,...,-,-,-,-,-,-,-,-,1011.7,embedded
2025-05-01 02:40:00,4.01,-0.14,82.4,1746085200,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,embedded
2025-05-01 03:10:00,4.01,-0.14,82.4,1746087000,0.628,6.827,3.291,73.11,67.33,57.726,...,-,-,-,-,-,-,-,-,1010.6,embedded
2025-05-01 04:40:00,3.99,-0.14,83.2,1746092400,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,embedded
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-05-31 19:10:00,4.04,-0.08,75.2,1748736600,0.582,8.533,3.373,111.922,73.695,55.895,...,-,-,-,-,-,-,-,-,1009.5,embedded
2025-05-31 20:40:00,4.04,-0.15,80.8,1748742000,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,embedded
2025-05-31 21:10:00,4.04,-0.15,80.8,1748743800,0.576,7.314,3.264,122.074,69.724,59.15,...,-,-,-,-,-,-,-,-,1010.9,embedded


# CTD sensors

In [32]:
ctd48m = ctd.CTD48M('../data/observations/CTD/CTD48M_SeaSunMarineTech.csv')
castaway = ctd.CastawayCTD('../data/observations/ctd/CC2349006_20240301_185500.csv')

In [33]:
ctd_objects = [ctd48m, castaway]

In [37]:
dict_ctd_raw_measurements = dict()
dict_ctd_clean_measurements = dict()

for ctd_obj in ctd_objects:
    raw_data = ctd_obj.get_raw_records()
    clean_data = ctd_obj.get_clean_records()
    dict_ctd_raw_measurements[ctd_obj.__class__.__name__] = raw_data
    dict_ctd_clean_measurements[ctd_obj.__class__.__name__] = clean_data

In [41]:
dict_ctd_clean_measurements['CTD48M']

,pressure[dbar],temperature[C],conductivity[uS/cm],specific_conductance[uS/cm],salinity[PSS],sound_velocity[m/s],density[kg/m3]
depth[m],,,,,,,
0.149915,0.150000,29.131853,58896.679687,54401.130407,36.009578,1544.844117,1022.781047
0.449744,0.450000,29.129037,58900.812264,54407.778109,36.014463,1544.848315,1022.786945
0.749572,0.750000,29.125827,58886.989676,54398.235556,36.007252,1544.838995,1022.783889
1.049400,1.050000,29.128301,58906.405764,54413.684773,36.018701,1544.861316,1022.792939
1.349227,1.350000,29.123365,58885.300862,54399.150344,36.007763,1544.844420,1022.787665
1.649055,1.650000,29.123191,58884.665700,54398.737991,36.007366,1544.848696,1022.788706
1.948882,1.950000,29.117810,58884.388325,54403.890349,36.011141,1544.846277,1022.794632
2.248708,2.250000,29.113707,58880.493284,54404.416588,36.011455,1544.842944,1022.797529
2.548532,2.550000,29.106853,58883.008041,54413.632366,36.018269,1544.840576,1022.806233


In [42]:
dict_ctd_clean_measurements['CastawayCTD']

,pressure[dbar],temperature[C],conductivity[uS/cm],specific_conductance[uS/cm],salinity[PSS],sound_velocity[m/s],density[kg/m3]
depth[m],,,,,,,
0.149818,0.150000,28.007169,58217.828125,54915.051157,36.394542,1542.819547,1023.444292
0.449452,0.450000,28.003554,58216.687377,54917.720512,36.396442,1542.818669,1023.448193
0.749086,0.750000,28.001095,58212.049293,54915.892851,36.394978,1542.816775,1023.449183
1.048720,1.050000,27.996325,58201.985534,54911.341388,36.391474,1542.807647,1023.449396
1.348354,1.350000,27.996636,58200.045029,54909.187730,36.389770,1542.811585,1023.449295
1.647988,1.650000,27.998245,58203.178329,54910.476570,36.390645,1542.821097,1023.450708
1.947621,1.950000,28.000806,58213.491300,54917.552840,36.395851,1542.837276,1023.455069
2.247254,2.250000,27.997998,58201.495832,54909.145562,36.389464,1542.829413,1023.452468
2.546887,2.550000,27.999634,58203.566994,54909.404951,36.389568,1542.838169,1023.453293


# Hobo sensor

# Mantaray